In [0]:
%pip install xgboost==3.3.0

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 MB 75.5 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.5/351.5 MB 115.3 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import os
import xgboost

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import PipelineModel
from pyspark.ml.linalg import Vectors
from pyspark.ml.functions import vector_to_array
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from xgboost.spark import SparkXGBClassifier

import os
import tempfile
import numpy as np
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import xgboost as xgb
print("XGBoost version:", xgboost.__version__)
print("Spark version:", spark.version)

XGBoost version: 3.3.0
Spark version: 4.2.0


In [0]:
import numpy as np
from xgboost import XGBClassifier

small_features = np.array([
    [0.0, 0.0],
    [0.1, 0.2],
    [0.8, 0.9],
    [1.0, 1.0]
], dtype=np.float32)

small_labels = np.array([0, 0, 1, 1])

small_model = XGBClassifier(
    n_estimators=2,
    max_depth=1,
    tree_method="hist",
    device="cpu",
    n_jobs=2,
    random_state=42
)

small_model.fit(small_features, small_labels)

print(small_model.predict_proba(small_features))
print("The Python XGBoost check passed.")

[[0.5 0.5]
 [0.5 0.5]
 [0.5 0.5]
 [0.5 0.5]]
The Python XGBoost check passed.


In [0]:
train = spark.table("workspace.mortgage_risk.model_train")
validation = spark.table("workspace.mortgage_risk.model_validation")

model_folder = "/Volumes/workspace/mortgage_risk/model_artifacts"
xgb_folder = model_folder + "/xgboost_python_v1"
os.makedirs(xgb_folder, exist_ok=True)

baseline_model = PipelineModel.load(
    model_folder + "/logistic_regression_v1"
)
preprocessing = PipelineModel(stages=baseline_model.stages[:-1])

preprocessing_path = xgb_folder + "/preprocessing"
model_path = xgb_folder + "/model.ubj"
train_path = xgb_folder + "/training_features"
validation_path = xgb_folder + "/validation_features"
score_path = xgb_folder + "/validation_scores.parquet"
xgb_score_table = "workspace.mortgage_risk.validation_scores_xgb_v1"


In [0]:
if os.path.exists(preprocessing_path):
    preprocessing = PipelineModel.load(preprocessing_path)
else:
    preprocessing.write().save(preprocessing_path)

print("Preprocessing ready.")


Preprocessing ready.


In [0]:
train_ready = preprocessing.transform(train).select(
    vector_to_array("features", "float32").alias("feature_values"),
    F.col("label_12m").cast("float").alias("label_12m")
)

if not os.path.exists(train_path):
    train_ready.write.mode("error").parquet(train_path)

print("Training features saved.")


Training features saved.


In [0]:
validation_ready = preprocessing.transform(validation).select(
    "loan_id",
    "prediction_month",
    "delinquency_months",
    "label_12m",
    vector_to_array("features", "float32").alias("feature_values")
)

if not os.path.exists(validation_path):
    validation_ready.write.mode("error").parquet(validation_path)

print("Validation features saved.")


Validation features saved.


In [0]:
train_data = ds.dataset(train_path, format="parquet")
validation_data = ds.dataset(validation_path, format="parquet")

training_rows = train_data.count_rows()
validation_rows = validation_data.count_rows()

assert training_rows == train.count(), "Training export is incomplete."
assert validation_rows == validation.count(), "Validation export is incomplete."
assert training_rows > 0 and validation_rows > 0

print("Training rows:", training_rows)
print("Validation rows:", validation_rows)


Training rows: 4299547
Validation rows: 1247263


In [0]:
class MortgageBatches(xgb.DataIter):
    def __init__(self, path, cache_prefix):
        self.data = ds.dataset(path, format="parquet")
        super().__init__(cache_prefix=cache_prefix)
        self.reset()

    def reset(self):
        self.batches = iter(self.data.to_batches(
            columns=["feature_values", "label_12m"],
            batch_size=100000,
            batch_readahead=1,
            fragment_readahead=1,
            use_threads=False
        ))

    def next(self, input_data):
        try:
            batch = next(self.batches)
        except StopIteration:
            return False

        values = batch.column(0).flatten().to_numpy(zero_copy_only=False)
        features = values.reshape(batch.num_rows, -1)
        labels = batch.column(1).to_numpy(zero_copy_only=False)

        input_data(data=features, label=labels)
        return True


In [0]:
parameters = {
    "objective": "binary:logistic",
    "tree_method": "hist",
    "device": "cpu",
    "max_depth": 4,
    "eta": 0.05,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "lambda": 1.0,
    "scale_pos_weight": 1.0,
    "max_bin": 128,
    "nthread": 2,
    "seed": 42
}


In [0]:
if os.path.exists(model_path):
    booster = xgb.Booster()
    booster.load_model(model_path)
    print("Loaded the saved XGBoost model.")
else:
    with tempfile.TemporaryDirectory(prefix="mortgage_xgb_") as cache_folder:
        batches = MortgageBatches(
            train_path,
            os.path.join(cache_folder, "training_cache")
        )
        training_matrix = xgb.ExtMemQuantileDMatrix(
            batches,
            max_bin=128,
            nthread=2
        )

        assert training_matrix.num_row() == training_rows

        booster = xgb.train(
            parameters,
            training_matrix,
            num_boost_round=200
        )
        booster.save_model(model_path)

        del training_matrix
        del batches

    print("XGBoost model saved.")


XGBoost model saved.


In [0]:
score_columns = [
    "loan_id", "prediction_month", "delinquency_months", "label_12m"
]
score_fields = []
for column in score_columns:
    score_fields.append(validation_data.schema.field(column))
score_fields.append(pa.field("risk_score", pa.float64()))
score_schema = pa.schema(score_fields)

if not os.path.exists(score_path):
    temporary_scores = xgb_folder + "/validation_scores.tmp.parquet"

    with pq.ParquetWriter(temporary_scores, score_schema) as writer:
        for batch in validation_data.to_batches(
            batch_size=100000,
            batch_readahead=1,
            fragment_readahead=1,
            use_threads=False
        ):
            feature_column = batch.schema.get_field_index("feature_values")
            values = batch.column(feature_column).flatten().to_numpy(
                zero_copy_only=False
            )
            features = values.reshape(batch.num_rows, -1)
            probabilities = booster.inplace_predict(features)

            scores = pa.Table.from_batches([batch]).select(score_columns)
            scores = scores.append_column(
                "risk_score",
                pa.array(probabilities, type=pa.float64())
            )
            writer.write_table(scores)

    os.replace(temporary_scores, score_path)

print("Validation scoring finished.")


Validation scoring finished.


In [0]:
xgb_scores = spark.read.parquet(score_path)

assert xgb_scores.count() == validation_rows, "Some predictions are missing."
invalid_scores = xgb_scores.filter(
    F.col("risk_score").isNull()
    | F.isnan("risk_score")
    | (F.col("risk_score") < 0)
    | (F.col("risk_score") > 1)
).count()
assert invalid_scores == 0, "Invalid prediction scores."

if spark.catalog.tableExists(xgb_score_table):
    xgb_scores = spark.table(xgb_score_table)
    print("Loaded the saved validation scores.")
else:
    xgb_scores.write.format("delta").mode("error").saveAsTable(
        xgb_score_table
    )
    xgb_scores = spark.table(xgb_score_table)
    print("Validation scores saved.")


Validation scores saved.


In [0]:
lr_scores = spark.table(
    "workspace.mortgage_risk.validation_scores_lr"
).withColumn("model", F.lit("Logistic regression"))

gbt_scores = spark.table(
    "workspace.mortgage_risk.validation_scores_gbt_v1"
).withColumn("model", F.lit("GBT: 50 trees"))

xgb_scores = xgb_scores.withColumn("model", F.lit("XGBoost"))

model_scores = lr_scores.unionByName(gbt_scores)
model_scores = model_scores.unionByName(xgb_scores)


In [0]:
pr_evaluator = BinaryClassificationEvaluator(
    labelCol="label_12m",
    rawPredictionCol="risk_score",
    metricName="areaUnderPR",
    numBins=10000
)

pr_rows = []

for model_name, scores in [
    ("Logistic regression", lr_scores),
    ("GBT: 50 trees", gbt_scores),
    ("XGBoost", xgb_scores)
]:
    pr_auc = pr_evaluator.evaluate(scores)
    pr_rows.append((model_name, pr_auc))

pr_results = spark.createDataFrame(pr_rows, ["model", "pr_auc"])


In [0]:
model_metrics = model_scores.groupBy("model").agg(
    F.count("*").alias("rows"),
    F.avg("label_12m").alias("event_rate"),
    F.avg("risk_score").alias("average_risk"),
    F.avg(
        F.pow(F.col("risk_score") - F.col("label_12m"), 2)
    ).alias("brier_score")
)

model_comparison = model_metrics.join(pr_results, "model")

display(
    model_comparison.select(
        "model", "rows", "pr_auc",
        "brier_score", "average_risk", "event_rate"
    ).orderBy("model")
)


model,rows,pr_auc,brier_score,average_risk,event_rate
GBT: 50 trees,1247263,0.17035239191121734,0.005231637010332688,0.031491337449977334,0.004917968383572671
Logistic regression,1247263,0.14747397306459945,0.004976100408080932,0.018023464847819852,0.004917968383572671
XGBoost,1247263,0.16343595894132115,0.004624779709968664,0.011431868585504395,0.004917968383572671


In [0]:
alert_scores = model_scores.withColumn(
    "population",
    F.lit("All eligible loans")
)

healthy_scores = alert_scores.filter(
    F.col("delinquency_months") == 0
).withColumn(
    "population",
    F.lit("Current status 00")
)

alert_scores = alert_scores.unionByName(healthy_scores)


In [0]:
month_window = Window.partitionBy(
    "model", "population", "prediction_month"
)

risk_window = month_window.orderBy(
    F.col("risk_score").desc(),
    F.xxhash64("loan_id", "prediction_month").asc(),
    F.col("loan_id").asc()
)

ranked = alert_scores.withColumn(
    "monthly_rows",
    F.count("*").over(month_window)
)

ranked = ranked.withColumn(
    "risk_rank",
    F.row_number().over(risk_window)
)


In [0]:
ranked = ranked.withColumn(
    "alert",
    F.when(
        F.col("risk_rank") <= F.ceil(F.col("monthly_rows") * 0.05),
        1
    ).otherwise(0)
)

ranked = ranked.withColumn(
    "event_caught",
    F.when(F.col("alert") == 1, F.col("label_12m"))
     .otherwise(0)
)


In [0]:
alert_comparison = ranked.groupBy(
    "model", "population"
).agg(
    F.count("*").alias("rows"),
    F.sum("label_12m").alias("events"),
    F.sum("alert").alias("alerts"),
    F.sum("event_caught").alias("events_caught")
)


In [0]:
alert_comparison = alert_comparison.withColumn(
    "precision",
    F.when(
        F.col("alerts") > 0,
        F.col("events_caught") / F.col("alerts")
    )
)

alert_comparison = alert_comparison.withColumn(
    "recall",
    F.when(
        F.col("events") > 0,
        F.col("events_caught") / F.col("events")
    )
)

display(
    alert_comparison.orderBy("population", "model")
)


model,population,rows,events,alerts,events_caught,precision,recall
GBT: 50 trees,All eligible loans,1247263,6134,62368,2864,0.04592098512057465,0.4669057711118357
Logistic regression,All eligible loans,1247263,6134,62368,2622,0.042040790148794256,0.4274535376589501
XGBoost,All eligible loans,1247263,6134,62368,2755,0.0441732939969215,0.44913596348223017
GBT: 50 trees,Current status 00,1239174,4250,61963,1053,0.016994012555880122,0.24776470588235294
Logistic regression,Current status 00,1239174,4250,61963,822,0.013265981311427788,0.19341176470588237
XGBoost,Current status 00,1239174,4250,61963,938,0.015138066265351904,0.22070588235294117


In [0]:
results_to_save = [
    (
        model_comparison,
        "workspace.mortgage_risk.model_comparison_xgb_v1"
    ),
    (
        alert_comparison,
        "workspace.mortgage_risk.alert_comparison_xgb_v1"
    )
]

for results, table_name in results_to_save:
    if spark.catalog.tableExists(table_name):
        print("Already saved:", table_name)
    else:
        results.write.format("delta").mode("error").saveAsTable(
            table_name
        )
        print("Saved:", table_name)


Saved: workspace.mortgage_risk.model_comparison_xgb_v1
Saved: workspace.mortgage_risk.alert_comparison_xgb_v1
